# Evaluating CFU model on entropy dataset

## Load data

Configure root.

In [ ]:
import sys, subprocess
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
%matplotlib inline
from pathlib import Path

# Configure root
COLAB = Path("/content").exists()
repo_url = "https://github.com/eddykang06/phenotype-prediction.git"
repo_dir = Path("phenotype-prediction")
if COLAB:
    root = Path("/content/phenotype-prediction")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", repo_url])
else:
    root = Path.cwd().parent
sys.path.insert(0, str(root))

Configure data path.

In [ ]:
if COLAB:
  from google.colab import drive
  drive.mount("/content/drive")
  data_dir = Path("/content/drive/MyDrive/phenotype-prediction-data")
  fcnts_path = str(data_dir / "fcnts_timezero")
  cfu_path = str(data_dir /  "cfus")
  annot_path = str(data_dir / "Annotation_TIGR4.tsv") 
  entropy_fcnts_path = str(data_dir / "entropy_data" / "fcnts")
  entropy_od_path = str(data_dir / "entropy_data" / "od600" / "growth_curves.csv")

else:
  data_dir = Path("C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab")
  fcnts_path = str(data_dir / "fcnts_timezero")
  cfu_path = str(data_dir / "cfus")
  annot_path = str(data_dir / "Annotation_TIGR4.tsv")
  entropy_fcnts_path = str(data_dir / "entropy_data" / "fcnts")
  entropy_od_path = str(data_dir / "entropy_data" / "od600" / "growth_curves.csv")
  entropy_path = data_dir / "entropy_data" / "entropy" / "entropy_values.csv"

Load TPM data from entropy dataset.

In [ ]:
from src.tpm_data import (
    fcnts_to_tpms, 
    read_fcnts_as_df, 
    bind_tpm_data,
    sample_name_strip,
    get_all_tpm_data
)

data_df = bind_tpm_data(fcnts_to_tpms(
    read_fcnts_as_df(entropy_fcnts_path, entropy = True),
    strip_leading_digits = True
))

Load original TPM data.

In [ ]:
og = get_all_tpm_data(
    fcnts_path = fcnts_path,
    cfu_path = cfu_path
)

# Find idx where CFU = 0, then list the sample ID
zero_idx = np.where(og["CFU"] == 0)[0]
print(og.index[zero_idx].tolist())

# Check CFU values for the other 2 replicates
rep_names = ["34CEF4hr-a", "34CEF4hr-b"]
cfu_val_check = og.loc[rep_names]["CFU"].tolist()
print(f"CFU values for 34CEF4hr-a and 34CEF4hr-b :{cfu_val_check}")

# Remove sample and convert to log 10 CFU
og = og[og["CFU"] != 0]
og["CFU"] = np.log10(og["CFU"])

Load OD600 data and convert to CFUs.

In [ ]:
import pandas as pd

df = pd.read_csv(entropy_od_path, header=[0, 1], index_col = 0)

df.index.name = "time_min"
df.columns.names = ["drug", "replicate"]

long_df = (
    df.stack(["drug", "replicate"], future_stack = True)
      .rename("OD600")
      .dropna()
      .reset_index()
)

long_df["drug_id"] = (
    long_df["drug"]
    + long_df["time_min"].astype(str)
    + "min-"
    + long_df["replicate"]
)

long_df = long_df.set_index("drug_id")[["OD600"]]

# Convert OD to CFU
long_df["CFU"] = np.log10(long_df["OD600"] * 10**8)

# Bind to TPM data
data_df = pd.merge(data_df, long_df, left_index = True, right_index = True, how = "inner")

Load and extract entropy values.

In [ ]:
import string

entropy_path = data_dir / "entropy_data" / "entropy" / "entropy_values.csv"
entropy = pd.read_csv(entropy_path)

# Filter out unneede info
entropy = entropy.drop(columns = ["Survive", "Group", "MOA", "Prediction"])
mask = (entropy["Strain"] == "T4") & (entropy["Adapted"] == False) & (entropy["Concentration"] == "L")
entropy = entropy[mask]
entropy

# New column of naming
entropy["id"] = entropy["AB"] + entropy["Time"].astype(str) + "min"
entropy["AB"].unique()

n = 3
suffixes = list(string.ascii_lowercase[:n])

entropy = (
    entropy.loc[entropy.index.repeat(n)]
    .reset_index(drop = True)
)

suffix_column = np.tile(suffixes, len(entropy) // n)

entropy["id"] = (
    entropy["id"].astype(str)
    + "-"
    + suffix_column
)
entropy = entropy.set_index("id")
entropy = entropy["Entropy"]

Extract metadata.

In [ ]:
from src.metadata import condition_to_drug_id, condition_to_timepoint

index = data_df.index
meta = pd.DataFrame(
    {
    "drug_id": [condition_to_drug_id(x) for x in index],
    "timepoint": [condition_to_timepoint(x) for x in index],
    "drug1_dose": [1]*len(index)
    }, 
    index = index
)

## Basic data viz

In [ ]:
# Examine growth curves over time
long_index = long_df.index
long_meta = pd.DataFrame(
    {
    "drug_id": [condition_to_drug_id(x) for x in long_index],
    "timepoint": [condition_to_timepoint(x) for x in long_index],
    "drug1_dose": [1]*len(long_index),
    "replicate": [x[-1] for x in long_index]
    }, 
    index = long_index
)
growth = pd.merge(long_df, long_meta, left_index = True, right_index = True, how = "inner")

fig, ax = plt.subplots(
    figsize = (15, 15), 
    nrows = 4, 
    ncols = 4,
    sharey = True
)
ax = ax.ravel()

for i, drug in enumerate(growth["drug_id"].unique()):
    df = growth[growth["drug_id"] == drug]
    sns.lineplot(
        data = df,
        x = "timepoint",
        y = "OD600",
        estimator = None,
        units = "replicate",
        ax = ax[i]
    )
    ax[i].set_title(drug)
    ax[i].set_xlabel("Time (min)")
plt.tight_layout()

## Model predictions on OD600 and entropy

Load model.

In [ ]:
# Get path to models
model_path = root / "models" / "diagonal_cfu_model.pkl"
forward_model = joblib.load(model_path)

Predictions on 1x MIC training data.

In [ ]:
from sklearn.metrics import r2_score

train_mask = (og["num_drugs"] == 1) | ((og["drug1_dose"]) == (og["drug2_dose"]))
train = og[train_mask]
train = train[train["drug1_dose"] == 1]

X_old = train.iloc[:, train.columns.str.contains("SP")]
y_old = train["CFU"]

meta_old = train.iloc[:, ~train.columns.str.contains("SP")].drop(columns = ["CFU"])

preds_old = forward_model.predict(X_old)

res_old = pd.DataFrame({
    "True log10 CFU": y_old,
    "Predicted log10 CFU": preds_old
})
res_old = pd.merge(res_old, meta_old, left_index = True, right_index = True, how = "left")
sns.scatterplot(
    data = res_old,
    x = "True log10 CFU",
    y = "Predicted log10 CFU",
    hue = "drug_id"
)
r2 = r2_score(res_old["True log10 CFU"], res_old["Predicted log10 CFU"])
plt.title(f"Model predictions on 1x MIC training data ($R^2$ = {r2:.3f})")

Predictions compared to OD600.

In [ ]:
import seaborn as sns
import scipy.stats as stats

X_test = data_df.iloc[:, data_df.columns.str.contains("SP")]
y_test = data_df.iloc[:, ~data_df.columns.str.contains("SP")]

preds = forward_model.predict(X_test)

res = pd.DataFrame({
    "True OD600": y_test["OD600"],
    "True log10 CFU": y_test["CFU"],
    "Predicted log10 CFU": preds
})
res = pd.merge(res, meta, left_index = True, right_index = True, how = "left")
res = pd.merge(res, entropy, left_index = True, right_index = True, how = "left")
sns.scatterplot(
    data = res,
    x = "True OD600",
    y = "Predicted log10 CFU",
    hue = "drug_id"
)
corr1 = stats.spearmanr(res["True OD600"], res["Predicted log10 CFU"])
plt.title(f"Model predictions for entropy paper dataset (Spearman r = {corr1.statistic:.3f})")

Predictions compared to entropy.

In [ ]:
sns.scatterplot(
    data = res,
    x = "Entropy",
    y = "Predicted log10 CFU",
    hue = "drug_id"
)
corr2 = stats.spearmanr(res["Entropy"], res["Predicted log10 CFU"])
plt.title(f"Model predictions for entropy paper dataset (Spearman r = {corr2.statistic:.3f})")